In [2]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 6 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# - maximise raw objective
# - use all observations through Week 7
# - fit ARD Matern GP
# - optimise GP hyperparameters automatically
# - local + wide + global candidate search
# - EI primary
# - UCB used to inspect exploration/exploitation

In [3]:
X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

assert len(X) == len(Y)
assert X.shape[1] == 5

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nY range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (27, 5)
Y shape: (27,)

Current best observed input:
[0.442929 0.409333 0.63183  0.7398   0.129381]

Current best observed output:
-0.19517557780237

Y range:
min = -2.5711696316081234
max = -0.19517557780237
std = 0.6272848999436226


In [4]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
1.22**2 * Matern(length_scale=[0.749, 0.884, 1.24, 0.859, 0.954], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[0.74886407 0.8838297  1.23984578 0.85869826 0.95356782]

Normalised inverse-lengthscale sensitivity:
[0.24338521 0.20621896 0.14700412 0.21225435 0.19113736]


In [6]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)


Local widths: [0.1 0.1 0.1 0.1 0.1]
Wide widths: [0.2 0.2 0.2 0.2 0.2]


In [7]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(80000, 5)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(50000, 5)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(100000, 5)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

print("Candidates before filtering:", len(candidates))

Candidates before filtering: 230000


In [8]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 229998


In [9]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("Predictions complete.")

Predictions complete.


In [10]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [11]:
EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY EI RESULT")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


PRIMARY EI RESULT
candidate = [0.46997634 0.45661268 0.586059   0.74329241 0.15001691]
mean = -0.19860998094440108
std = 0.028185693207809115
EI = 0.009610634538024798


In [12]:
y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI sensitivity check:

xi = 0.000000e+00 
 candidate = [0.46997634 0.45661268 0.586059   0.74329241 0.15001691] 
 mean = -0.19861 
 std = 0.028186 
 EI = 0.00961063 

xi = 6.272849e-03 
 candidate = [0.47626262 0.46722844 0.6480119  0.76945708 0.16748581] 
 mean = -0.212279 
 std = 0.041936 
 EI = 0.00757565 

xi = 3.136424e-02 
 candidate = [0.46246078 0.51193195 0.60488888 0.77736072 0.16791201] 
 mean = -0.235404 
 std = 0.05732 
 EI = 0.00290577 

xi = 6.272849e-02 
 candidate = [0.01723165 0.0031298  0.43441029 0.00248416 0.03377845] 
 mean = -1.468038 
 std = 0.543475 
 EI = 0.00124153 



In [13]:
print("\nUCB diagnostic:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [0.46182595 0.44166    0.61348803 0.74469234 0.14666184] 
 mean = -0.192818 
 std = 0.02027 
 UCB = -0.190791 

beta=0.25 
 candidate = [0.46182595 0.44166    0.61348803 0.74469234 0.14666184] 
 mean = -0.192818 
 std = 0.02027 
 UCB = -0.187751 

beta=0.5 
 candidate = [0.46182595 0.44166    0.61348803 0.74469234 0.14666184] 
 mean = -0.192818 
 std = 0.02027 
 UCB = -0.182683 

beta=1.0 
 candidate = [0.48958224 0.4519432  0.64416069 0.75795356 0.16118448] 
 mean = -0.207431 
 std = 0.037251 
 UCB = -0.17018 



In [14]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.46697038 0.40661431 0.59750597 0.74804058 0.15268504]
mean = -0.19241318078159897
std = 0.013842155508492182


In [15]:
# --------------------------------------------------
# Final Function 6 Week 8 selection
# --------------------------------------------------
#
# The GP posterior mean and low-to-moderate UCB
# settings identify the same local region.
#
# beta = 0.1, 0.25 and 0.5 all select the same point,
# showing that the recommendation is stable across
# a reasonable exploration range.
#
# beta = 0.5 is retained as the balanced
# exploration-exploitation setting.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week8_candidate = candidates[final_idx]

print("Week 8 Function 6 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 6 candidate:
[0.46182595 0.44166    0.61348803 0.74469234 0.14666184]

Predicted mean:
-0.19281840768046887

Predicted std:
0.020270458244346076

UCB:
-0.18268317855829583

Portal format:
0.461826-0.441660-0.613488-0.744692-0.146662
